# 🎬 Movie Recommendation System

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 1. Load Data

In [2]:
data = pd.read_csv('dataset.csv')

In [3]:
data.head()

,id,title,genre,original_language,overview,popularity,release_date,vote_average,vote_count
0,278,The Shawshank Redemption,"Drama,Crime",en,Framed in the 1940s for the double murder of h...,94.075,1994-09-23,8.7,21862
1,19404,Dilwale Dulhania Le Jayenge,"Comedy,Drama,Romance",hi,"Raj is a rich, carefree, happy-go-lucky second...",25.408,1995-10-19,8.7,3731
2,238,The Godfather,"Drama,Crime",en,"Spanning the years 1945 to 1955, a chronicle o...",90.585,1972-03-14,8.7,16280
3,424,Schindler's List,"Drama,History,War",en,The true story of how businessman Oskar Schind...,44.761,1993-12-15,8.6,12959
4,240,The Godfather: Part II,"Drama,Crime",en,In the continuing saga of the Corleone crime f...,57.749,1974-12-20,8.6,9811


In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 10000 non-null  int64  
 1   title              10000 non-null  object 
 2   genre              9997 non-null   object 
 3   original_language  10000 non-null  object 
 4   overview           9987 non-null   object 
 5   popularity         10000 non-null  float64
 6   release_date       10000 non-null  object 
 7   vote_average       10000 non-null  float64
 8   vote_count         10000 non-null  int64  
dtypes: float64(2), int64(2), object(5)
memory usage: 703.3+ KB


In [5]:
data.isnull().sum()

id                    0
title                 0
genre                 3
original_language     0
overview             13
popularity            0
release_date          0
vote_average          0
vote_count            0
dtype: int64

## 2. Feature Engineering

We combine `genre` and `overview` into a single `tags` column. We must:
- Fill nulls before concatenating (otherwise NaN propagates)
- Add a space between genre and overview so words don't merge

In [6]:
# Fill nulls before concatenation — otherwise NaN spreads to the whole tag
data['genre'] = data['genre'].fillna('')
data['overview'] = data['overview'].fillna('')

# Add a space separator so genre words don't merge with overview words
# e.g. "Drama,Crime" + "Framed..." → "Drama Crime Framed..." (clean)
data['genre_clean'] = data['genre'].str.replace(',', ' ')

data['tags'] = data['genre_clean'] + ' ' + data['overview']
data['tags'] = data['tags'].str.lower().str.strip()

In [7]:
data[['title', 'tags']].head()

,title,tags
0,The Shawshank Redemption,drama crime framed in the 1940s for the double...
1,Dilwale Dulhania Le Jayenge,"comedy drama romance raj is a rich, carefree, ..."
2,The Godfather,"drama crime spanning the years 1945 to 1955, a..."
3,Schindler's List,drama history war the true story of how busine...
4,The Godfather: Part II,drama crime in the continuing saga of the corl...


## 3. Select Relevant Columns

In [8]:
new_data = data[['id', 'title', 'tags']].reset_index(drop=True)

In [9]:
new_data.head()

,id,title,tags
0,278,The Shawshank Redemption,drama crime framed in the 1940s for the double...
1,19404,Dilwale Dulhania Le Jayenge,"comedy drama romance raj is a rich, carefree, ..."
2,238,The Godfather,"drama crime spanning the years 1945 to 1955, a..."
3,424,Schindler's List,drama history war the true story of how busine...
4,240,The Godfather: Part II,drama crime in the continuing saga of the corl...


## 4. Vectorize Tags

In [11]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(max_features=10000, stop_words='english')
vector = cv.fit_transform(new_data['tags'].values.astype('U'))
print("Vector shape:", vector.shape)

Vector shape: (10000, 10000)


## 5. Compute Cosine Similarity

In [12]:
from sklearn.metrics.pairwise import cosine_similarity

sim = cosine_similarity(vector)
print("Similarity matrix shape:", sim.shape)

Similarity matrix shape: (10000, 10000)


## 6. Recommend Function

In [13]:
def recommend(movie_name):
    
    matches = new_data[new_data['title'].str.lower() == movie_name.lower()]
    if matches.empty:
        return f"Movie '{movie_name}' not found in dataset."

    movie_index = matches.index[0]

    distances = sorted(
        list(enumerate(sim[movie_index])),
        reverse=True,
        key=lambda x: x[1]
    )

    recommendations = []
    for movie in distances[1:6]:   # skip index 0 (the movie itself)
        recommendations.append(new_data.iloc[movie[0]].title)

    return recommendations

## 7. Test Recommendations

In [14]:
recommend("Iron Man")

['Iron Man 3',
 'Guardians of the Galaxy Vol. 2',
 'Avengers: Age of Ultron',
 'Star Wars: Episode III - Revenge of the Sith',
 'Iron Man 2']

In [15]:
recommend("The Shawshank Redemption")

['Brubaker', 'The Woodsman', 'A Prophet', 'Gotti', 'Cool Hand Luke']

In [16]:
recommend("The Godfather")

['The Godfather: Part II', 'Blood Ties', 'Joker', 'Bomb City', 'Gotti']

## 8. Save Artifacts

In [17]:
import joblib

joblib.dump(cv, "vectorizer.pkl")
joblib.dump(sim, "similarity.pkl")
joblib.dump(new_data, "movies.pkl")

print("Saved: vectorizer.pkl, similarity.pkl, movies.pkl")

Saved: vectorizer.pkl, similarity.pkl, movies.pkl
